# Solutions · Chapter 04-08 · Reproducibility

Worked answers for `notebooks/04_workflow/04-08_reproducibility.ipynb`.

E9 and E12 both make a prediction and then check it, which is the habit this whole module has been
teaching.

In [ ]:
import json
import platform
import subprocess
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# SYNTHETIC: the chapter's data
rng = np.random.default_rng(11)
n_rows = 800
features = pd.DataFrame({"x1": rng.normal(size=n_rows),
                         "x2": rng.normal(size=n_rows),
                         "x3": rng.normal(size=n_rows)})
target = (2 * features.x1 - 1.5 * features.x2
          + 0.5 * features.x1 * features.x2 + rng.normal(0, 1.0, n_rows)).to_numpy()


def score(split_seed, model_seed):
    train_index, test_index = train_test_split(np.arange(n_rows), test_size=0.3,
                                               random_state=split_seed)
    forest = RandomForestRegressor(n_estimators=50, min_samples_leaf=3, random_state=model_seed)
    forest.fit(features.iloc[train_index], target[train_index])
    return mean_absolute_error(target[test_index], forest.predict(features.iloc[test_index]))


split_varies = np.array([score(seed, 0) for seed in range(30)])
print("set up. split-seed spread: mean %.4f, sd %.4f" % (split_varies.mean(), split_varies.std()))

## Quick understanding

### E1

| Level | Fixes |
|---|---|
| **1 Seeds** | the same answer twice, on this machine, today, with these libraries |
| **2 Code** | that somebody runs the same steps - a commit, not "the notebook" |
| **3 Data** | that the same rows go in - a snapshot or an as-of date |
| **4 Environment** | that the same arithmetic comes out - pinned versions |
| **5 Hardware** | that the last few decimals agree - threads, BLAS, GPU |

### E2

**The split seed, by a factor of 5.2** - standard deviation 0.0487 against the model seed's 0.0094.

The reason is that the split changes *which rows the model is scored on*, and rows differ from each other
far more than a forest's internal bootstrap draws differ from each other. Varying the model seed rebuilds
the same model from the same data slightly differently; varying the split seed asks a different question.

### E3

Any two of: **thread count** (`n_jobs` or BLAS threads change summation order), **library versions** (the
algorithm may differ), **GPU reductions** (non-deterministic by default), **Python hash randomisation**
(set and dict iteration order), and **data updated in place** (the rows are simply different).

## Hand calculation

### E4

`(0.8809 - 0.7859) / 0.0487` = **1.95 standard deviations below the mean.**

**What to say:** that 0.7859 is a real number honestly computed, and also the best of what appears to be a
search over seeds - it sits at the bottom of a distribution whose spread we have measured. The reportable
figure is the mean with its spread, **0.8809 +/- 0.0487**, and if a single split must be quoted then the
seed has to be fixed *before* the results are seen, not chosen after.

Said more usefully: **"which seeds did you try?"** If the answer is "just that one", it is a lucky draw
and worth re-running. If the answer is "all of them, and I picked the best", that is 04-03's selection
premium with the candidates being seeds.

### E5

`1.192e-07 / 1.08e+08` = **1.10e-15**.

Float64 carries about 15 to 17 significant decimal digits, so a relative difference of 1e-15 is **the last
digit**. You can trust roughly **fifteen significant figures** of that total and no more - which is far
more precision than any measurement in this course requires, and is why the chapter calls this effect
irrelevant on its own.

### E6

The expected minimum of 30 draws is about 2.04 standard deviations below the mean, so reporting the best
of 30 seeds gives about

`0.8809 - 2.04 x 0.0487` = **0.7814**

against an honest mean of **0.8809**. That is an improvement of 0.10, or **11%**, obtained by typing
thirty numbers and keeping one.

The measured minimum over the thirty seeds was 0.7859, close to the prediction - which is a small
confirmation that the scores behave like independent draws from a roughly normal distribution.

### E7

`4^12` = **16,777,216 distinct environments.**

That is the point of pinning: an unpinned requirements file does not describe an environment, it describes
a set of sixteen million of them, and you have tested exactly one.

In [ ]:
print("E4  (0.8809 - 0.7859) / 0.0487 = %.2f standard deviations below the mean"
      % ((0.8809 - 0.7859) / 0.0487))
print("E5  1.192e-07 / 1.08e+08 = %.2e  -> about 15 significant figures" % (1.192e-07 / 1.08e08))
print("E6  best of 30 expected: %.4f, actually observed: %.4f, honest mean: %.4f"
      % (split_varies.mean() - 2.04 * split_varies.std(), split_varies.min(), split_varies.mean()))
print("E7  4 ** 12 = %s environments" % format(4 ** 12, ","))

## Coding

### E8 - seeding everything

In [ ]:
import random


def set_all_seeds(seed):
    random.seed(seed)              # the standard library
    np.random.seed(seed)           # numpy's legacy global state
    return np.random.default_rng(seed)   # and a generator to pass around


generator = set_all_seeds(7)
print("stdlib random :", round(random.random(), 6))
print("numpy legacy  :", round(float(np.random.rand()), 6))
print("generator     :", round(float(generator.random()), 6))

**Why the modern advice is to pass a generator explicitly.** `np.random.seed` sets *global* state, which
means:

- **Any library you call can consume from it**, advancing the stream in ways you cannot see. Add a call to
  a function that happens to sample internally, and every subsequent draw changes.
- **It does not compose.** Two parts of a program that both seed the global generator fight each other,
  and the winner depends on import and call order.
- **It is not thread-safe or process-safe.** Parallel workers inherit or re-seed it unpredictably.

Passing a `Generator` object makes the dependency explicit: a function that needs randomness takes one as
an argument, so you can see from the signature that it is random, and two callers cannot interfere. This
course uses `default_rng` everywhere for exactly that reason.

The legacy seeding is still in `set_all_seeds` because third-party code may use it, and you cannot always
tell which.

### E9 - a model with no randomness

**Predict first:** `Ridge` has no `random_state` with the default solver. So the model-seed spread should
be **exactly zero**, and the split-seed spread should look much like the forest's.

In [ ]:
def ridge_score(split_seed):
    train_index, test_index = train_test_split(np.arange(n_rows), test_size=0.3,
                                               random_state=split_seed)
    fitted = Ridge(alpha=1.0).fit(features.iloc[train_index], target[train_index])
    return mean_absolute_error(target[test_index], fitted.predict(features.iloc[test_index]))


ridge_split = np.array([ridge_score(seed) for seed in range(30)])
repeated = np.array([ridge_score(0) for _ in range(10)])

print("ridge, split seed varies : mean %.4f  sd %.4f" % (ridge_split.mean(), ridge_split.std()))
print("ridge, nothing varies    : sd %.1e over 10 refits" % repeated.std())
print()
print("forest, split seed varies: mean %.4f  sd %.4f" % (split_varies.mean(), split_varies.std()))

**Confirmed: the model-seed spread is zero for any practical purpose** - ten refits of ridge on
identical data agree to about 2e-16, the last bit of a float64, because the fit is a closed-form
linear-algebra solution with nothing random in it.

And the split-seed spread is **0.0513**, close to the forest's 0.0487. **Almost all of the variation in a
reported score comes from which rows you held out, not from the model** - and that is true even for a
model with no randomness at all.

This is worth stating as the chapter's real conclusion: **a deterministic model does not give you a
deterministic result.** It gives you a deterministic result *per split*, and the split is the part that
moves.

### E10 - capturing and comparing environments

In [ ]:
def capture_environment():
    import os
    return {"python": sys.version.split()[0],
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__,
            "platform": "%s %s" % (platform.system(), platform.machine()),
            "cpus": os.cpu_count()}


def compare_environments(before, after):
    keys = sorted(set(before) | set(after))
    differences = [(key, before.get(key, "absent"), after.get(key, "absent"))
                   for key in keys if before.get(key) != after.get(key)]
    if not differences:
        print("environments are identical")
        return
    print("%-14s %-18s %-18s" % ("", "before", "after"))
    for key, old, new in differences:
        print("%-14s %-18s %-18s  <-- differs" % (key, old, new))


current = capture_environment()
pretended = dict(current, **{"scikit-learn": "1.12.0", "cpus": 4})
compare_environments(current, pretended)

Printing only the differences is the whole design. A full environment dump is dozens of lines that nobody
reads; **two lines saying what changed is a diagnosis.**

### E11 - an experiment log

In [ ]:
LOG = "experiments.jsonl"


def log_experiment(name, why, settings, scores, path=LOG):
    try:
        commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True, timeout=5).stdout.strip()
    except Exception:
        commit = "unknown"
    entry = {"experiment": name, "why": why, "code": commit or "uncommitted",
             "settings": settings,
             "result": round(float(np.mean(scores)), 4),
             "spread": round(float(np.std(scores)), 4),
             "runs": len(scores),
             "environment": capture_environment()}
    with open(path, "a") as handle:
        handle.write(json.dumps(entry) + "\n")
    return entry


def load_experiments(path=LOG):
    with open(path) as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    frame = pd.json_normalize(rows)
    return frame.sort_values("result")


import os
if os.path.exists(LOG):
    os.remove(LOG)

for leaf_size in [1, 3, 10]:
    scores = [score(seed, 0) for seed in range(10)]
    forest_scores = []
    for seed in range(10):
        train_index, test_index = train_test_split(np.arange(n_rows), test_size=0.3, random_state=seed)
        fitted = RandomForestRegressor(n_estimators=50, min_samples_leaf=leaf_size,
                                       random_state=0).fit(features.iloc[train_index], target[train_index])
        forest_scores.append(mean_absolute_error(target[test_index],
                                                 fitted.predict(features.iloc[test_index])))
    log_experiment("forest-leaf-%d" % leaf_size,
                   "does a larger leaf help on data with an interaction?",
                   {"min_samples_leaf": leaf_size, "n_estimators": 50}, forest_scores)

table = load_experiments()
print(table[["experiment", "why", "result", "spread", "runs"]].to_string(index=False))
os.remove(LOG)

Three runs, one line of JSON each, and a table that sorts. **The `why` column is what makes a log of forty
of these navigable**, and it is the field that gets dropped first.

Note that the spreads (around 0.05) are larger than the differences between the leaf sizes. **The log
records that, so a reader can see the comparison is inconclusive** - which a table of three bare numbers
would have hidden.

### E12 - does `n_jobs` change the answer?

In [ ]:
train_index, test_index = train_test_split(np.arange(n_rows), test_size=0.3, random_state=0)

one_thread = RandomForestRegressor(200, random_state=0, n_jobs=1)
one_thread.fit(features.iloc[train_index], target[train_index])
all_threads = RandomForestRegressor(200, random_state=0, n_jobs=-1)
all_threads.fit(features.iloc[train_index], target[train_index])

single = one_thread.predict(features.iloc[test_index])
parallel = all_threads.predict(features.iloc[test_index])

print("same random_state, different n_jobs:")
print("  predictions bit-identical : %s" % np.array_equal(single, parallel))
print("  largest difference        : %.3e" % np.abs(single - parallel).max())
print("  MAE difference            : %.3e"
      % abs(mean_absolute_error(target[test_index], single)
            - mean_absolute_error(target[test_index], parallel)))

**Not bit-identical - the largest difference is of order 1e-15**, and it will not be the same figure
every time you run this, because it depends on how the threads happened to divide the work.

The trees themselves are identical: `random_state=0` fixes every bootstrap sample and every split. What
differs is the **order in which 200 tree predictions are averaged**, which changes with how the work was
divided between threads - and floating-point addition is not associative.

**The size tells you it does not matter.** A difference of order 1e-15 on predictions of order 1 is the
last bit of a float64, and the MAE difference is smaller still. If your conclusion depends on that, your conclusion was not
robust to begin with.

**When it does matter:** when tiny differences get amplified by a branch. A tie broken differently in a
tree split, a threshold crossed at exactly 0.5, an early-stopping rule that fires one iteration later -
these turn a last-bit difference into a visibly different model. That is why deep-learning frameworks ship
explicit deterministic modes, and why those modes are slower.

## Interpretation

### E13

What I would need, in order of how likely each is to explain a 0.3% difference:

1. **The variance.** Run-to-run and split-to-split spread. In this chapter the split seed alone moved the
   score by 5%, seventeen times the claimed improvement. **This is far and away the most likely
   explanation**, and it is the one a single number cannot rule out.
2. **How many variants were tried.** 04-03's arithmetic: the best of twenty attempts is inflated by about
   1.9 standard errors even when nothing real is happening.
3. **Whether the comparison is paired** - same splits, same folds, same preprocessing. An unpaired 0.3%
   is much weaker evidence than a paired one (04-07's E11 measured that difference directly).
4. **The environment**, if the baseline number was taken from a previous paper rather than re-run. A
   library upgrade between the two is entirely capable of 0.3%.
5. **The seeds**, last, because a missing seed produces noise of the kind item 1 already covers.

The honest short version: **a 0.3% improvement reported without a spread is not a measurement.**

### E14

**Explanation 1: the data is drifting.** The world changed - customer mix, seasonality, a new product -
so the model trained on recent data faces a harder problem. **Check:** plot the target's base rate and the
feature distributions by month. This is the most likely cause of a *slow, steady* drift.

**Explanation 2: the label quality or availability changed.** A logging change, a delayed label, an
upstream job that started dropping rows. **Check:** row counts and null rates per run, from the pipeline's
own logs. A step change hiding inside a smooth-looking average is common.

**Explanation 3: the environment moved under you.** An unpinned dependency was upgraded by a base-image
rebuild. **Check:** diff the captured environment record between the first and the latest run - which is
free if you captured it, and impossible if you did not.

**"Nothing in the code changed" is exactly why this is hard**, and it is the argument for recording the
other four levels. Three of these three explanations are invisible from the code.

## Debugging

### E15

**Where to look:** the environment first - library versions and thread counts - and then whether either
machine has a different BLAS or CPU architecture. A third-decimal difference is the signature of
**arithmetic**, not of logic.

**Where not to bother:** the code, the data and the seeds. If a seed were missing you would see run-to-run
variation on a *single* machine too, which is a one-line check; and if the code or data differed the
discrepancy would almost certainly be larger than the third decimal.

Then ask the question that usually ends it: **does it matter?** If the conclusion is the same on both
machines, a third-decimal difference is a curiosity. Chasing bit-identical results across architectures is
expensive and is rarely what anybody actually needed.

### E16

**A difference in the split, the data, or the metric - not in the arithmetic.**

If both machines reproduce their own numbers exactly, then the *procedures* differ rather than the
execution of one procedure. The candidates, in order:

- **Different splits.** Different `random_state`, or a different splitting *strategy* - one grouped, one
  not. 04-04 showed that alone reversing a model ranking.
- **Different data.** One of you has an older snapshot, or a filter applied earlier in the notebook.
- **Different metric or averaging.** MAE against RMSE, macro against micro, or a mean over folds against
  a score on pooled predictions.

**"Both reproduce, but disagree" is a much better position to be in than "neither reproduces"** - two
deterministic procedures can be diffed, and the diff is guaranteed to explain the disagreement.

## Exam and interview reasoning

### E17

> "Seeds are the easy part and the smallest part. I set `random_state` on every estimator and splitter,
> but that only guarantees I get my own number back today. Above that: the code has to be a commit, not a
> notebook state; the data has to be a snapshot or an as-of query, because tables get updated in place;
> and the environment has to be pinned, because a library upgrade can change a default and move the result
> with no error message. I also capture the versions automatically into an experiment record alongside the
> result, the spread and the baseline - so when a number moves, I can diff rather than guess. And I report
> a spread, because a stable number and a representative one are different things: on the data I was
> working with, the choice of split seed moved the score by five percent."

**"We do not have time for all that - what is the minimum?"**

> "Three things, and they take about ten minutes. Pin your dependency versions. Seed your splits. Report a
> cross-validated mean and spread instead of one number. The first prevents the failure that has no error
> message, the second makes today repeatable, and the third stops you over-reading a difference smaller
> than your own noise. Everything else - commits, data snapshots, experiment logs - is worth adding as the
> project gets more valuable, but those three are cheap enough that there is no version of 'not enough
> time' that excludes them."

## Transfer to a different situation

### E18

What must travel with the fitted pipeline:

| Item | Why |
|---|---|
| **The serialised pipeline** | it carries the imputer, scaler, encoder and model together (04-07) |
| **The exact library versions it was fitted with** | unpickling across versions is unsupported and can fail silently |
| **The expected input schema** - column names, dtypes, units | a `ColumnTransformer` addresses by name; a renamed column is a silent failure |
| **The training data's ranges and category lists** | so drift and unseen categories can be monitored (04-06's E12) |
| **The evaluation record** - metric, value, spread, baseline, split rule | so "is it still working?" has a reference point |
| **The prediction contract** from 04-01 | unit, target, prediction time, horizon - so nobody calls it for the wrong question |
| **A worked example** - one input row and its expected output | the fastest possible smoke test after any deployment |

The last one is worth insisting on: **a single input-output pair catches version mismatches, column
reordering and serialisation problems in one second**, and it costs one row in a JSON file.

## Explain it to someone non-technical

### E19

> "A recipe that says 'bake until done' works fine for the person who wrote it, because they know what
> done looks like. Hand it to somebody else and you get a different cake. Our analysis is the same: it
> depended on things I never wrote down - which version of each tool I had installed, exactly which rows I
> pulled that morning, and a few choices I made without noticing. Writing all of that down alongside the
> result is what turns 'it worked on my laptop' into something the team can check and build on."

(89 words.)

## Optional challenge

### E20 - a regression test for a result

In [ ]:
class ResultHarness:
    def __init__(self, tolerance=1e-9):
        self.log = {}
        self.tolerance = tolerance

    def run(self, name, experiment, settings):
        signature = json.dumps({"name": name, "settings": settings}, sort_keys=True)
        value = float(experiment(**settings))
        if signature not in self.log:
            self.log[signature] = value
            print("%-22s recorded  %.10f" % (name, value))
            return value
        previous = self.log[signature]
        if abs(previous - value) <= self.tolerance:
            print("%-22s matches   %.10f" % (name, value))
        else:
            print("%-22s CHANGED   was %.10f, now %.10f  (difference %.2e)"
                  % (name, previous, value, abs(previous - value)))
        return value


def experiment(min_samples_leaf, model_seed):
    train_index, test_index = train_test_split(np.arange(n_rows), test_size=0.3, random_state=0)
    fitted = RandomForestRegressor(50, min_samples_leaf=min_samples_leaf,
                                   random_state=model_seed).fit(features.iloc[train_index],
                                                                target[train_index])
    return mean_absolute_error(target[test_index], fitted.predict(features.iloc[test_index]))


harness = ResultHarness()
settings = {"min_samples_leaf": 3, "model_seed": 0}

harness.run("baseline", experiment, settings)                    # first time: recorded
harness.run("baseline", experiment, settings)                    # second time: must match
harness.log[json.dumps({"name": "baseline", "settings": settings}, sort_keys=True)] += 0.01
harness.run("baseline", experiment, settings)                    # tampered: must be caught
print()
harness.run("bigger-leaf", experiment, {"min_samples_leaf": 10, "model_seed": 0})

**Three behaviours, all demonstrated:** an unseen configuration is recorded, a repeat of it must match to
1e-9, and a stored value that has drifted is reported as changed rather than silently overwritten. A
different configuration gets its own entry, because the signature includes the settings.

**This is a regression test for a *result*, and almost no project has one.** Codebases are full of tests
asserting that functions return the right types and that pipelines do not crash; very few assert that the
number the project exists to produce is still the number it produced last month.

Three things to be careful about if you build a real one:

- **The tolerance is a modelling decision.** 1e-9 catches everything including harmless thread-order
  differences, which will make it noisy across machines. A tolerance of 1e-6, or "within a tenth of the
  measured fold spread", is usually more useful - and choosing it forces you to say how much change would
  actually matter.
- **The signature must include the environment**, or the test will pass on your machine and fail in CI
  for a reason it cannot report. Add `capture_environment()` to the key, or at least record it alongside.
- **A failing result test is information, not a bug.** It fires when you improve the model too. Its job is
  to make sure the change was *intended*, which is exactly the job of any regression test.

That is the last idea in module 04, and it closes the loop the module opened: **04-01 said a model is a
means to a decision. This says the number behind that decision should be as defended as the code that
produced it.**